In [ ]:
%%configure  

{ 
    "vCores": 
    { 
        "parameterName": "pipelinecore", 
        "defaultValue": 2 
    }
}

In [ ]:
!pip install -q duckrun --upgrade
notebookutils.session.restartPython()


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    lakehouse_name = vl.lakehouse_name
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
    token          = notebookutils.credentials.getToken('storage')
    dbt_target     = 'dev'
    notebookutils.fs.mount(
        f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}',
        '/lh',
        {'fileCacheTimeout': 0},
    )
    mount_path = notebookutils.fs.getMountPath('/lh')
    dbt_path   = f'{mount_path}/Files/dbt'
except ModuleNotFoundError:
    from azure.identity import AzureCliCredential
    import yaml
    from pathlib import Path
    _root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "deploy_config.yml").exists()), None)
    if _root is None:
        raise FileNotFoundError("deploy_config.yml not found in cwd or any parent — run from inside the cloned repo")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws"]
    lakehouse_id   = _cfg["lakehouse"]
    lakehouse_name = _cfg["lakehouse_name"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    token          = AzureCliCredential().get_token("https://storage.azure.com/.default").token
    dbt_target     = 'dev'
os.environ['download_limit']   = download_limit
os.environ['process_limit']    = process_limit

In [ ]:
os.environ['FILES_PATH']          = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files'
os.environ['ONELAKE_TABLES_PATH'] = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables'
os.environ['ONELAKE_TOKEN']       = token

In [ ]:
from dbt.cli.main import dbtRunner
os.chdir(dbt_path)
dbt = dbtRunner()
import datetime
# One timestamped log folder per run, beside the dbt project (NOT inside it) on the OneLake mount so logs survive a crash.
_log_dir = f"{os.path.dirname(dbt_path)}/dbt_runs_logs/run-{datetime.datetime.now():%Y%m%dT%H%M%S}"
base = ["--target", dbt_target, "--profiles-dir", ".",
        "--log-path", _log_dir, "--target-path", "/tmp/dbt_target"]

# 1. Download / refresh the archive log first.
dbt.invoke(["run", "--select", "stg_csv_archive_log", *base])

# 2. New daily files pending? check_new_daily "fails" when yes -- checked BEFORE fct_scada ingests them.
new_daily = not dbt.invoke(["run-operation", "check_new_daily", *base]).success
print(f"new_daily = {new_daily}")

# 3. Build everything except the summary (retry failures once).
#    No new daily landed -> the daily fct_price/fct_scada have nothing to do, so exclude
#    them entirely. Selecting them would still scan the full table just to discover there's
#    nothing new -- this skips the node, zero work.
daily_ex = [] if new_daily else ["--exclude", "fct_price", "--exclude", "fct_scada"]
result = dbt.invoke(["run", "--exclude", "stg_csv_archive_log", "--exclude", "fct_summary", *daily_ex, *base])
if not result.success:
    print("dbt run had failures -- retrying failed models once...")
    _ = dbt.invoke(["retry", *base])

# 4. Summary: overwrite when a new daily landed, else append intraday.
summary = ["run", "--select", "fct_summary", *base] + (["--full-refresh"] if new_daily else [])
_ = dbt.invoke(summary)

# 5. Tests.
_ = dbt.invoke(["test", *base])